# AstroCLIMB — three-seed ensemble + modality mask

CPU-only postprocessing notebook for the equal-probability Qwen3-VL-8B ensemble of seeds **17, 42, and 123**. It applies exactly one change: `same_figure` is assigned zero probability to caption–caption and image–image pairs, after which the remaining probabilities are renormalized.

Attach the AstroCLIMB competition data and the three-seed ensemble output containing `submission_probabilities.csv`. No model, adapter, GPU, swap TTA, calibration, or restricted-loss blending is used.

In [ ]:
import csv
import json
import math
import sys
from collections import Counter
from pathlib import Path

csv.field_size_limit(sys.maxsize)

TARGETS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
PROBABILITY_COLUMNS = [f'p_{name}' for name in TARGETS]
EXPECTED_TEST_ROWS = 10_000

# Set explicit paths if automatic discovery finds multiple attached datasets.
ENSEMBLE_PROBABILITIES = '/kaggle/input/datasets/syedmohaiminulhoque/submission-probabilities/submission_probabilities.csv'
TEST_CSV = None
# Optional JSONL produced by the training notebooks. When supplied, it avoids
# streaming the large test.csv merely to recover object modalities.
TEST_MANIFEST = None

WORK_ROOT = (
    Path('/kaggle/working/astroclimb_qwen3vl8b_three_seed_modality_mask')
    if Path('/kaggle/working').exists()
    else Path('./astroclimb_qwen3vl8b_three_seed_modality_mask')
)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [p for p in [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')] if p.exists()]
print('Output:', WORK_ROOT)

## Locate and validate inputs

The probability file must contain raw four-class probabilities. A hard one-hot `submission.csv` is not a valid input.

In [ ]:
def normalized_path(path):
    return str(path).lower().replace('-', '_')

def discover_ensemble_probabilities():
    candidates = []
    for root in SEARCH_ROOTS:
        for path in root.rglob('submission_probabilities.csv'):
            name = normalized_path(path)
            if (
                'three_seed' in name
                and 'ensemble' in name
                and not any(term in name for term in ('modality_mask', 'swap_tta', 'blend'))
            ):
                candidates.append(path)
    candidates = sorted(set(candidates), key=lambda p: (len(str(p)), str(p)))
    if len(candidates) != 1:
        raise RuntimeError(
            'Expected exactly one unprocessed three-seed ensemble probability file; '
            f'found {len(candidates)}. Set ENSEMBLE_PROBABILITIES explicitly. Candidates: {candidates}'
        )
    return candidates[0]

def discover_test_csv():
    preferred = [
        Path('/kaggle/input/competitions/astroclimb/test.csv'),
        Path('/kaggle/input/astroclimb/test.csv'),
        Path('data/test.csv'),
    ]
    for path in preferred:
        if path.is_file():
            return path
    candidates = [p for root in SEARCH_ROOTS for p in root.rglob('test.csv')]
    candidates = sorted(set(candidates), key=lambda p: ('astroclimb' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError('AstroCLIMB test.csv was not found. Set TEST_CSV explicitly.')
    return candidates[0]

def read_probabilities(path):
    path = Path(path)
    if path.name == 'submission.csv':
        raise ValueError('Use raw submission_probabilities.csv, not a one-hot submission.csv')
    rows = {}
    order = []
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        expected = ['id', *PROBABILITY_COLUMNS]
        if reader.fieldnames != expected:
            raise ValueError(f'{path}: expected columns {expected}, got {reader.fieldnames}')
        for row in reader:
            row_id = str(row['id'])
            if row_id in rows:
                raise ValueError(f'{path}: duplicate id {row_id}')
            values = [float(row[column]) for column in PROBABILITY_COLUMNS]
            if not all(math.isfinite(value) and value >= 0 for value in values):
                raise ValueError(f'{path}: invalid probabilities for id {row_id}')
            total = sum(values)
            if not math.isclose(total, 1.0, abs_tol=1e-5):
                raise ValueError(f'{path}: probabilities do not sum to one for id {row_id}: {total}')
            rows[row_id] = values
            order.append(row_id)
    if len(rows) != EXPECTED_TEST_ROWS:
        raise ValueError(f'{path}: expected {EXPECTED_TEST_ROWS} rows, got {len(rows)}')
    return rows, order

probability_path = Path(ENSEMBLE_PROBABILITIES) if ENSEMBLE_PROBABILITIES else discover_ensemble_probabilities()
test_path = None if TEST_MANIFEST else (Path(TEST_CSV) if TEST_CSV else discover_test_csv())
ensemble, ordered_ids = read_probabilities(probability_path)
print('Ensemble probabilities:', probability_path)
print('Test modality source:', TEST_MANIFEST or test_path)
print('Probability rows:', len(ensemble))

## Stream object modalities

Image payloads are recognized by common encoded-image signatures. The large CSV strings are inspected but never decoded or retained in memory.

In [ ]:
IMAGE_PREFIXES = ('iVBORw0KGgo', '/9j/', 'UklGR', 'R0lGOD', 'data:image')

def looks_like_image(value):
    return isinstance(value, str) and value.lstrip().startswith(IMAGE_PREFIXES)

def modalities_from_csv(path):
    modalities = {}
    with Path(path).open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        missing = {'id', 'obj_1', 'obj_2'} - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'{path}: missing columns {sorted(missing)}')
        for index, row in enumerate(reader, start=1):
            row_id = str(row['id'])
            if row_id in modalities:
                raise ValueError(f'{path}: duplicate id {row_id}')
            kind_1 = 'I' if looks_like_image(row['obj_1']) else 'C'
            kind_2 = 'I' if looks_like_image(row['obj_2']) else 'C'
            modalities[row_id] = kind_1 + kind_2
            if index % 1000 == 0:
                print(f'Inspected {index} test rows', flush=True)
    return modalities

def modalities_from_manifest(path):
    modalities = {}
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            row = json.loads(line)
            row_id = str(row['id'])
            if row_id in modalities:
                raise ValueError(f'{path}: duplicate id {row_id}')
            modality = row.get('modality')
            if modality not in {'CC', 'CI', 'IC', 'II'}:
                kind_1 = 'I' if row['obj_1']['kind'] == 'image' else 'C'
                kind_2 = 'I' if row['obj_2']['kind'] == 'image' else 'C'
                modality = kind_1 + kind_2
            modalities[row_id] = modality
    return modalities

modalities = (
    modalities_from_manifest(TEST_MANIFEST)
    if TEST_MANIFEST
    else modalities_from_csv(test_path)
)
if set(modalities) != set(ensemble):
    raise ValueError(
        f'ID mismatch: probabilities-only={len(set(ensemble)-set(modalities))}, '
        f'modalities-only={len(set(modalities)-set(ensemble))}'
    )
modality_counts = Counter(modalities.values())
if not set(modality_counts).issubset({'CC', 'CI', 'IC', 'II'}):
    raise ValueError(f'Unexpected modalities: {modality_counts}')
print('Modality counts:', dict(sorted(modality_counts.items())))
print('Rows to mask:', modality_counts['CC'] + modality_counts['II'])

## Apply the mask and write outputs

Only `CC` and `II` rows are changed. Cross-modal `CI` and `IC` probabilities remain numerically identical to the raw ensemble.

In [ ]:
probability_output = WORK_ROOT / 'submission_probabilities.csv'
submission_output = WORK_ROOT / 'submission.csv'
manifest_output = WORK_ROOT / 'modality_manifest.csv'
report_output = WORK_ROOT / 'mask_report.json'

raw_counts = Counter()
masked_counts = Counter()
changed_predictions = 0
invalid_raw_same_figure = 0

with (
    probability_output.open('w', encoding='utf-8', newline='') as probability_handle,
    submission_output.open('w', encoding='utf-8', newline='') as submission_handle,
    manifest_output.open('w', encoding='utf-8', newline='') as manifest_handle,
):
    probability_writer = csv.writer(probability_handle, lineterminator='\n')
    submission_writer = csv.writer(submission_handle, lineterminator='\n')
    manifest_writer = csv.writer(manifest_handle, lineterminator='\n')
    probability_writer.writerow(['id', *PROBABILITY_COLUMNS])
    submission_writer.writerow(['id', *TARGETS])
    manifest_writer.writerow(['id', 'modality', 'same_figure_masked', 'raw_prediction', 'masked_prediction'])

    for row_id in ordered_ids:
        raw = ensemble[row_id]
        raw_prediction = max(range(4), key=raw.__getitem__)
        raw_counts[TARGETS[raw_prediction]] += 1
        modality = modalities[row_id]
        should_mask = modality in {'CC', 'II'}
        masked = list(raw)
        if should_mask:
            if raw_prediction == 0:
                invalid_raw_same_figure += 1
            masked[0] = 0.0
            remaining_mass = sum(masked)
            if remaining_mass <= 0:
                raise ValueError(f'No probability mass remains after masking id {row_id}')
            masked = [value / remaining_mass for value in masked]
        masked_prediction = max(range(4), key=masked.__getitem__)
        masked_counts[TARGETS[masked_prediction]] += 1
        changed_predictions += int(masked_prediction != raw_prediction)
        one_hot = [int(index == masked_prediction) for index in range(4)]
        probability_writer.writerow([row_id, *[f'{value:.16g}' for value in masked]])
        submission_writer.writerow([row_id, *one_hot])
        manifest_writer.writerow([
            row_id, modality, int(should_mask), TARGETS[raw_prediction], TARGETS[masked_prediction]
        ])

report = {
    'experiment': 'three_seed_ensemble_modality_mask_only',
    'ensemble_probability_source': str(probability_path),
    'modality_source': str(TEST_MANIFEST or test_path),
    'rows': len(ordered_ids),
    'modality_counts': dict(sorted(modality_counts.items())),
    'masked_rows': modality_counts['CC'] + modality_counts['II'],
    'raw_invalid_same_figure_predictions': invalid_raw_same_figure,
    'changed_predictions': changed_predictions,
    'raw_prediction_counts': dict(raw_counts),
    'masked_prediction_counts': dict(masked_counts),
    'swap_tta': False,
    'calibration': False,
    'restricted_loss_blend': False,
}
report_output.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))

## Validate final artifacts

In [ ]:
def validate_outputs(probability_path, submission_path):
    with Path(probability_path).open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ['id', *PROBABILITY_COLUMNS]:
            raise ValueError(f'Unexpected probability columns: {reader.fieldnames}')
        probability_rows = list(reader)
    with Path(submission_path).open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ['id', *TARGETS]:
            raise ValueError(f'Unexpected submission columns: {reader.fieldnames}')
        submission_rows = list(reader)
    if len(probability_rows) != EXPECTED_TEST_ROWS or len(submission_rows) != EXPECTED_TEST_ROWS:
        raise ValueError('Output row count is not 10,000')
    probability_ids = [row['id'] for row in probability_rows]
    submission_ids = [row['id'] for row in submission_rows]
    if probability_ids != ordered_ids or submission_ids != ordered_ids:
        raise ValueError('Output IDs do not preserve ensemble order')
    if len(set(submission_ids)) != EXPECTED_TEST_ROWS:
        raise ValueError('Output IDs are not unique')
    for probability_row, submission_row in zip(probability_rows, submission_rows):
        values = [float(probability_row[column]) for column in PROBABILITY_COLUMNS]
        if not all(math.isfinite(value) and value >= 0 for value in values):
            raise ValueError(f'Invalid output probabilities for id {probability_row["id"]}')
        if not math.isclose(sum(values), 1.0, abs_tol=1e-10):
            raise ValueError(f'Output probabilities do not sum to one for id {probability_row["id"]}')
        targets = [int(submission_row[column]) for column in TARGETS]
        if not all(value in (0, 1) for value in targets) or sum(targets) != 1:
            raise ValueError(f'Invalid one-hot output for id {submission_row["id"]}')
        modality = modalities[probability_row['id']]
        if modality in {'CC', 'II'} and values[0] != 0.0:
            raise ValueError(f'same_figure was not masked for id {probability_row["id"]}')
    return True

assert validate_outputs(probability_output, submission_output)
print('Validated probabilities:', probability_output)
print('Validated submission:', submission_output)
print('Modality manifest:', manifest_output)
print('Mask report:', report_output)